In [ ]:
import os 
os.environ["OMP_NUM_THREADS"] = "50"

In [ ]:
import numpy as np
import healpy as hp
import astropy.units as u

import torch
from scipy.ndimage import gaussian_filter

import matplotlib.pyplot as plt
import scienceplots
plt.style.use(["science"])

plt.rcParams.update({'font.size': 12})
plt.rcParams.update({
    "text.usetex": True,        
    "font.family": "serif",     
    "font.serif": ["Times"],
    "figure.dpi": 300,          
    "axes.grid": True,          
    "grid.linestyle": "--",     
})
from pathlib import Path
from ds_utils import  *
from ps_utils import  *

In [ ]:
DATA_DIR = Path("/pscratch/sd/k/kp22/deft/")

In [ ]:
RES = 256
STEP_SIZE = 6 * u.deg
ANG_X = 6. * u.deg
ANG_Y = 6. * u.deg
ptsrc = 2
flatskymapparams = [RES, RES, 60.*ANG_X.value/RES, 60.*ANG_Y.value/RES] #Code requires pixelres to be in arcmin
flatskymapparams

# Creating training data  (Skip this section if the maps are already saved)

## 1. Loading the point source masked maps
These maps are same as the AGORA maps but the point sources are masked at a threshold of 2mJ/6mJy and those points are replaced by zeroes.

These files are available at: /sptlocal/analysis/ymap/sims/mdpl2/data/v0.7/bahamas80_scal1.000/mask_radio_cib_6.0mjy/cib

## 2. Preprocessing the maps
Before we create training patches, we need to low pass filter these maps so that we don't have higher frequency modes aliasing into lower frequency modes. To do this we apply a sharp cutoff for modes above 7000. 
I noticed that there are a few pixels that have negative values that doesn't make physical sense. I believe they are a result of some filtering process. We can safely zero them out as they are only a fraction of a percent.

## 3. Creating map patches
The patches are created by laying down a set of centroids $(l_i + s, b_i +\frac{s}{\cos(l_i)})$ (longitude,latitude), where $s$ is a step size parameter, and $\frac{s}{\cos(l_i)}$ is a step between longitudes for a given latitude, which ensures the same angular separation in the latitudinal direction. Each centroid is then rotated to the equator, and an $6 \times 6$ square degrees region around the centroid is projected onto a cartesian grid with 256 pixels along each size.

### a. Loading the point source masked maps

In [ ]:
cib_150_map = hp.read_map(Path(DATA_DIR/ f"mask_radio_cib_{ptsrc}mJy" / "mdpl2_150GHz_fullsky.fits"))
#cib_150_map = hp.read_map(Path(DATA_DIR/ f"mask_radio_cib_{ptsrc}mJy" / "tsz" / "mdpl2_150GHz_fullsky_mask_3e+14.fits"))

### b. Low pass filtering

In [ ]:
fullsky_map = np.copy(cib_150_map)

In [ ]:
hp.mollview(cib_150_map, title="", min=0, max=60, unit=r"$\mu K$", cmap='Blues')
fig = plt.gcf()
cax = fig.axes[-1]
cax.tick_params(labelsize=20)
#plt.savefig("figures/cib_150_map.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
fullsky_alm = hp.map2alm(fullsky_map,lmax=13000)
ell_cut = 7000
lmax=13000
alm_filtered = hp.sphtfunc.almxfl(fullsky_alm, [1 if ell <= ell_cut else 0 for ell in range(lmax + 1)])

In [ ]:
fullsky_map = hp.sphtfunc.alm2map(alm_filtered, nside=8192)

In [ ]:
fullsky_map[fullsky_map<0]=0 # For CIB
#fullsky_map[fullsky_map>0]=0 # For tSZ

In [ ]:
hp.mollview(fullsky_map, title="", min=0, max=60, unit=r"$\mu K$", cmap='Blues')
fig = plt.gcf()
cax = fig.axes[-1]
cax.tick_params(labelsize=20,)
#cax.set_ylabel(r"$\mu K$", fontsize=20, loc="bottom")

#plt.savefig("figures/cib_150_map_processed.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
# Doing this at this stage is bad because it can't be inverted from cut maps. Should have done it later after map cuts
cib_150_map_unfiltered=np.copy(cib_150_map)
ell_cut = 7000
lmax=13000
alm_filtered = hp.sphtfunc.almxfl(cib_map_8192_alm, [1 if ell <= ell_cut else 0 for ell in range(lmax + 1)])

In [ ]:
cib_150_map = hp.sphtfunc.alm2map(alm_filtered, nside=8192)

In [ ]:
np.min(cib_150_map),np.max(cib_150_map),np.mean(cib_150_map)

In [ ]:
cib_150_map -= np.min(cib_150_map) 

In [ ]:
cib_150_map = np.log10(1+cib_150_map)

In [ ]:
np.min(cib_150_map),np.max(cib_150_map),np.mean(cib_150_map)

In [ ]:
hp.mollview(cib_150_map,title="",unit=r"$\mu$K",min=-60,max=0, cmap='viridis',cbar=True)
#plt.savefig("figures/cib_150_map_processed.png",bbox_inches="tight",dpi=200)

### c. Creating map patches

In [ ]:
# get centroids
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))

In [ ]:
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, fullsky_map) for (lon, lat) in centers], dtype=np.float32)

In [ ]:
fname = f"data/low_pass/{ptsrc}mJy/cut_maps_RES_{RES}_ANG_X_{ANG_X}_{ptsrc}mJy_lp_cib150.npy"
np.save(fname,cut_maps)

## 4. MinMax Normalization

Let each map be denoted by $m(x, y)$. 
We normalize (MinMax) the spatial domain maps to have values between 0 and 1. Here $\vec{m}$ denotes the collection of maps.
$$m = \frac{m - \min(\vec{m})}{\max(\vec{m}) - \min(\vec{m})}$$

In [ ]:
cut_maps= np.load(fname)

In [ ]:
def apply_maxmin_normalization(maps):
    min_val = np.min(maps)
    max_val = np.max(maps)
    return (maps - min_val) / (max_val - min_val) 

In [ ]:
processed_maps = np.copy(cut_maps)

In [ ]:
min_val = np.min(cut_maps)
max_val = np.max(cut_maps)
print (min_val,max_val - min_val)  

In [ ]:
processed_maps = apply_maxmin_normalization(processed_maps)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(processed_maps[21], cmap='Blues', vmin=0, vmax=1)
ax.set_xlabel(r"$6^\circ$", fontsize=20)
ax.set_ylabel(r"$6^\circ$", fontsize=20)
ax.grid()

cbar = fig.colorbar(im, ax=ax, orientation='horizontal',
                    fraction=0.046)
cbar.set_ticks([0, 1])
cbar.ax.tick_params(labelsize=20)

#cbar.set_ticks([0, 0.5, 1])

ax.tick_params(axis='both', which='both',
               bottom=False, top=False,
               left=False, right=False,
               labelbottom=False, labelleft=False)

#plt.savefig("figures/cib_150_map_flatsky.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
fpath = "data/low_pass/{:d}mJy/CIB_map_150GHz_{:d}_st{:d}_minmax_{:d}mJy_zero_lp.npy".format(ptsrc,RES, int(STEP_SIZE.value),ptsrc)
#fpath = "data/low_pass/{:d}mJy/tSZ3_map_150GHz_{:d}_st{:d}_minmax_{:d}mJy_norm_lp.npy".format(ptsrc,RES, int(STEP_SIZE.value),ptsrc)
print(fpath)

In [ ]:
np.save(fpath, processed_maps)

# Flowchart

In [ ]:
import matplotlib.patches as patches

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.image as mpimg

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

img_fullsky1 = mpimg.imread('figures/cib_150_map.png')
img_fullsky2 = mpimg.imread('figures/cib_150_map_processed.png')
img_flatsky  = mpimg.imread('figures/cib_150_map_flatsky.png')

# Define node positions and sizes: (x, y, width, height)
nodes = {
    'node1': (0.05, 0.35, 0.1, 0.3),
    'node2': (0.43, 0.35, 0.1, 0.3),
    'node3': (0.67, 0.35, 0.25, 0.3),
}

def place_image(img, coords, zoom=0.1):
    x = coords[0] + coords[2] / 2
    y = coords[1] + coords[3] / 2
    im = OffsetImage(img, zoom=zoom)
    ab = AnnotationBbox(im, (x, y), frameon=False)
    ax.add_artist(ab)

# Place images
place_image(img_fullsky1, nodes['node1'], zoom=0.15)
place_image(img_fullsky2, nodes['node2'], zoom=0.15)
place_image(img_flatsky,  nodes['node3'], zoom=0.15)

# Calculate bottom center for each node (i.e., the lower edge)
node1_bottom = (nodes['node1'][0] + nodes['node1'][2] / 2, nodes['node1'][1])
node2_bottom = (nodes['node2'][0] + nodes['node2'][2] / 2, nodes['node2'][1])
node3_bottom = (nodes['node3'][0] + nodes['node3'][2] / 2, nodes['node3'][1])

# Add an extra downward offset to position the arrows further below the images
arrow_offset = -0.3
arrow_start12 = (node1_bottom[0] +0.1 , node1_bottom[1] + arrow_offset)
arrow_end12   = (node2_bottom[0]-0.1, node2_bottom[1] + arrow_offset)
arrow_start23 = (node2_bottom[0]+0.05, node2_bottom[1] + arrow_offset)
arrow_end23   = (node3_bottom[0]-0.05, node3_bottom[1] + arrow_offset)

# Draw arrow from node1 to node2 with labels below the arrow
ax.annotate("", xy=arrow_end12, xytext=arrow_start12, 
            arrowprops=dict(arrowstyle="->", lw=1, shrinkA=5, shrinkB=5))
midpoint12 = ((arrow_start12[0] + arrow_end12[0]) / 2, (arrow_start12[1] + arrow_end12[1]) / 2)
ax.text(midpoint12[0], midpoint12[1] + 0.05, "2mJy point source mask", ha='center', fontsize=12)
ax.text(midpoint12[0], midpoint12[1] - 0.09, "Low pass filter at 7000", ha='center', fontsize=12)

# Draw arrow from node2 to node3 with labels below the arrow
ax.annotate("", xy=arrow_end23, xytext=arrow_start23, 
            arrowprops=dict(arrowstyle="->", lw=1, shrinkA=5, shrinkB=5))
midpoint23 = ((arrow_start23[0] + arrow_end23[0]) / 2, (arrow_start23[1] + arrow_end23[1]) / 2)
ax.text(midpoint23[0], midpoint23[1] + 0.05, "Extracting flatsky patches", ha='center', fontsize=12)
ax.text(midpoint23[0], midpoint23[1] - 0.09, "Normalization 0 to 1", ha='center', fontsize=12)
plt.savefig("figures/map_processing.png", bbox_inches="tight", dpi=200)
plt.savefig("figures/map_processing.pdf", bbox_inches="tight", dpi=200)
plt.show()


# Gaussian patches

In [ ]:
cib_map_meansub = cib_150_map - np.mean(cib_150_map)
cl_from_map = hp.anafast(cib_map_meansub, lmax=13000)

In [ ]:
gaussian_map = hp.synfast(cl_from_map, 8192)

In [ ]:
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))

In [ ]:
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, gaussian_map) for (lon, lat) in centers], dtype=np.float32)

In [ ]:
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_tsz.npy",cut_maps)

# Joint gaussian maps

In [ ]:
cib_map_meansub = cib_150_map - np.mean(cib_150_map)
cib_cl_from_map = hp.anafast(cib_map_meansub, lmax=13000)

In [ ]:
tsz_map_meansub = tsz_150_map - np.mean(tsz_150_map)
tsz_cl_from_map = hp.anafast(tsz_map_meansub, lmax=13000)

In [ ]:
hp.write_cl("data/low_pass/2mJy/anafast_tsz3_cl_from_map.fits",tsz_cl_from_map)

In [ ]:
hp.write_cl("data/low_pass/2mJy/anafast_cib_cl_from_map.fits",cib_cl_from_map)

In [ ]:
cib_tsz_cl_from_map = hp.anafast(cib_map_meansub,tsz_map_meansub, lmax=13000)

In [ ]:
hp.write_cl("data/low_pass/2mJy/anafast_cib_tsz3_cl_from_map.fits",cib_tsz_cl_from_map)

In [ ]:
cib_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_cib_cl_from_map.fits")

In [ ]:
tsz_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_tsz3_cl_from_map.fits")

In [ ]:
cib_tsz_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_cib_tsz3_cl_from_map.fits")

In [ ]:
# Ordering for healpy.synfast:
# [C_ℓ^TT (aa), C_ℓ^EE (bb), C_ℓ^BB (ignored), C_ℓ^TE (ab)]
cls = [cib_cl_from_map, tsz_cl_from_map, np.zeros_like(cib_cl_from_map), cib_tsz_cl_from_map]
maps = hp.synfast(cls, nside=8192, new=True, pol=False)
cib_map = maps[0]
tsz_map = maps[1]

In [ ]:
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, cib_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_cib_joint3.npy",cut_maps)

In [ ]:
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, tsz_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_tsz_joint3.npy",cut_maps)

# Save fg cl

In [ ]:
fpath = "data/low_pass/6mJy/cut_maps_RES_256_ANG_X_6.0 deg_6mJy_lp.npy"
print(fpath)
cut_maps = np.load(fpath)

In [ ]:
el, cl_mean  = np.mean([map2cl(flatskymapparams, cut_maps[i,:,:,0]) for i in range(len(cut_maps))], axis=0)

In [ ]:
from scipy.interpolate import interp1d
interp_fn = interp1d(el, cl_mean, kind='cubic', bounds_error=False, fill_value='extrapolate')
el_full, cl_cib = np.loadtxt(DATA_DIR/f"data/cl_cib.csv",unpack=True, delimiter=',')
el_int = np.arange(0, 24576)
cl_cib_int = interp_fn(el_full)

In [ ]:
data = np.column_stack((el_int, cl_cib_int))

In [ ]:
np.savetxt(DATA_DIR/f"data/mean_cl_cib_patch_6mJy.csv", data, delimiter=",")

In [ ]:
plt.loglog(el, cl_mean,label="Mean")
plt.plot(el_int, cl_cib_int,'--')
#plt.plot(el_full, cl_cib, alpha=0.4)

plt.legend()
plt.xlim(10,10000)
#plt.ylim(5e-6,5e-4)